In [3]:
def frac(x):
    return(x-ZZ(floor(x)))

Given $\omega\in K$ a real quadratic number, $c\geq0,d\in\mathbb{Z}^2$, let $\beta=c\omega+d$, we want to compute
\begin{equation}
\begin{split}
h_\omega(t,c,d,v)&=\sum_{j=0}^{c-1}f_t\biggl(\biggl\{v_1-\frac{d}{c}(j+v_2)\biggl\}\biggl)f_{\beta t}\biggl(\frac{j+\{v_2\}}{c}\biggl)
\\
=&\sum_{j=0}^{c-1}\frac{e^{t\{v_1-\frac{d}{c}(j+v_2)\}}}{(e^t-1)}\frac{e^{\beta t\bigl(\frac{j+\{v_2\}}{c}\bigl)}}{(e^{\beta t}-1)},
\end{split}
\end{equation}
We first compute independently 

$$tf_t\biggl(\biggl\{v_1-\frac{d}{c}(j+v_2)\biggl\}\biggl)= \frac{te^{t\{v_1-\frac{d}{c}(j+v_2)\}}}{e^t-1}
\,\,\,\textup{and }\,\,\,
tf_{ t}\biggl(\frac{j+\{v_2\}}{c}\biggl)=\frac{te^{ t\frac{j+\{v_2\}}{c}}}{e^{ t}-1}.$$

As polynomials in $\mathbb{Q}$, and then we remark that 

$$
    f_t\biggl(\biggl\{v_1-\frac{d}{c}(j+v_2)\biggl\}\biggl) f_{\beta t}\biggl(\frac{j+\{v_2\}}{c}\biggl)[t^k]=\sum_{j=-1}^{k+1}\beta^jf_t\biggl(\frac{j+\{v_2\}}{c}\biggl)[t^j]f_t\biggl(\biggl\{v_1-\frac{d}{c}(j+v_2)\biggl\}\biggl) [t^{k-j}].
$$

To stay in $K[[t]]$ before the smoothing, we actually compute $t^2h_\omega(t,c,d,v)$, this explain that you will see an offset of 2 when we take the coeffcients of $h_\omega(t,c,d,v)$ in the code below.

In [4]:



def power_series_values_for_sum(c,d,order,v1,v2):##we dont put beta to optimize the computation
    Pol.<u> = QQ[]   
    L1=[]
    L2=[]
    order=order+1
    den = u._exp_series(order+1) >> 1 # shift (P-P(0))/u
    invden = den.inverse_series_trunc(order) ## use the flint library
    for j in range(0,c):  
        x1 = Pol.base_ring()((j+frac(v2))/c) 
        num1 = (x1*u)._exp_series(order) #_exp_series result as a polynomial
        res1 = num1.multiplication_trunc(invden, order)
        x2 = Pol.base_ring()(frac(v1-(d/c)*(j+frac(v2))))
        num2 = (x2*u)._exp_series(order) ## give e^(x2u) as a polynomial
        res2=num2.multiplication_trunc(invden, order) ## multiplication of polynomials calling Flint
        L1.append(res1);
        L2.append(res2);
    return([L1,L2])

def power_series_values_for_sum_norm_0(c,d,order,v1,v2):##same version but we change the value of b_1(0), needed for ord_p and log_beta.
    Pol.<u> = QQ[]   
    L1=[]
    L2=[]
    order=order+1
    den = u._exp_series(order+1) >> 1 # shift (P-P(0))/u
    invden = den.inverse_series_trunc(order) ## use the flint library, compute 1/den
    for j in range(0,c):  
        x1 = Pol.base_ring()((j+frac(v2))/c) 
        num1 = (x1*u)._exp_series(order) #_exp_series result as a polynomial
        res1 = num1.multiplication_trunc(invden, order)
        if frac(((j+frac(v2))/c))==0:## we change the value of b_1(0), 
            res1=res1+u/2
        x2 = Pol.base_ring()(frac(v1-(d/c)*(j+frac(v2))))
        num2 = (x2*u)._exp_series(order) ## give e^(x2u) as a polynomial
        res2=num2.multiplication_trunc(invden, order) ## multiplication of polynomials calling Flint
        if frac(v1-(d/c)*(j+frac(v2)))==0:## same thing, we change the value of b_1(0)
            res2=res2+u/2
        L1.append(res1);
        L2.append(res2);
    return([L1,L2])


def sum_for_H_value_at_k(beta,c,d,k,L1,L2):## last computation in K, using the power series computed before
    val=0
    for j in range (0,c):
        for r in range (0,k+1):
            val=val+(beta)^(r-1)*L1[j][r]*L2[j][k-r]
    return(val)

# Smoothing

Let $\ell$ be a prime number and suppose that $\ell$ divide $c$, we compute the smoothed version $h_\omega^{(\ell)}$ where
$$h_\omega^{(\ell)}(t,c,d,v):=h_{\ell\omega}\biggl(\frac{t}{\ell},\frac{c}{\ell},d,(\ell v_1,v_2)\biggl)-\ell h_\omega(t,c,d,(v_1,v_2)).$$

We simply call twice the functions above.

In [28]:
def power_series_values_for_sum_smoothed(c,d,order,l,v1,v2):
    t1=cputime()
    L=power_series_values_for_sum(c,d,order,v1,v2)
    L_smoothed=power_series_values_for_sum(c/l,d,order,l*v1,v2)
    return([L,L_smoothed])

def power_series_values_for_sum_smoothed_norm_0(c,d,order,l,v1,v2):
    t1=cputime()
    L=power_series_values_for_sum_norm_0(c,d,order,v1,v2)
    L_smoothed=power_series_values_for_sum_norm_0(c/l,d,order,l*v1,v2)
    return([L,L_smoothed])


def sum_for_H_smoothed_value_at_k(beta,c,d,k,l,L,L_smoothed):#L=[L1,L2]
    t1=cputime()
    val1=sum_for_H_value_at_k(beta,c,d,k,L[0],L[1])
    val2=sum_for_H_value_at_k(beta,c/l,d,k,L_smoothed[0],L_smoothed[1])
    val_final=-(l*val1-val2/l^(k-2))
    return(val_final)


# Computing the integral 


In the following, $(c,d)$ is the lower row of the stabilizer of $\omega$ in $\Gamma_0(\ell)$. We want to use the cocycle relation to work with $c=\ell$ instead, we write 
$$\Bigl\{\infty\rightarrow\frac{d}{c}\Bigl\}=\sum_{i,\gamma}a_{\gamma,i}\gamma\Bigl\{\infty\rightarrow\frac{i}{\ell}\Bigl\}.$$
The functions defined below with "cocycle" in the name use this decomposition. Thez will be used when $c$ is bigger than $\ell$ times the number of non-zero $a_{\gamma,i}$.

The factor 12 in the formulas is dropped when $\ell\geq 5$. It becomes a $3$ when $\ell=3$ and a $4$ when $\ell=2$

## Computing $\textup{ord}_p$

Using Equation (34):
$$\textup{ord}_p(u_{\ell}^\prime(\omega))=12\bigl(h_\omega^{(\ell)}(t,c,d,0,0)[t^0]+\biggl\lfloor\frac{\ell-1}{4}\biggl\rfloor\bigl)$$


In [2]:
def order_p_low_level(c,d,p,l): ## here we need b_1(0)=0
    L_both=power_series_values_for_sum_smoothed_norm_0(c,d,2,l,0,0) ##this is the constant term (recall the shift of 2)
    return((sum_for_H_smoothed_value_at_k(1,c,d,2,l,L_both[0],L_both[1])+(l-1)/4)) ## the constant term does not depend on the value so we can set x=1

def order_p(c,d,p,l):
    val=order_p_low_level(c,d,p,l)
    if l==2:
        return(ZZ(4*val))
    if l==3:
        return(ZZ(3*val))
    return(ZZ(val))

########################### using the cocycle relation
def order_p_cocycle(omega,Listmat,p,l,c_start,d_start):
    c=l
    val=0
    for mat in Listmat:
        Mat=Matrix([[mat.a,mat.b],[mat.c,mat.d]])
        if Mat.det()!=0:
            d=mat.x
            val+=order_p_low_level(c,d,p,l)*mat.coeff
            if sgn(d_start)>0 and sgn(mat.a*d+l*mat.b) == -1:## See the github of Fleischer and Liu for the details
                val += (l-1)/2
            elif sgn(d_start)<0 and sgn(mat.a*d+l*mat.b) == 1:
                val += (l-1)/2
    if sgn(d_start) < 0:
        val+= (l-1)/2
    if l==2:
        return(ZZ(4*val))
    if l==3:
        return(ZZ(3*val))
    return(ZZ(val))            

## Computing $\log_{\beta_p}$

We have
$$
   \log_{\beta_p}(u_{\ell}^\prime(\omega)) =12\sum_{\substack{a,b\mod p\\a,b\neq 0,0}}\log_{\beta_p}(a+b\omega)h_\omega^{(\ell)}\biggl(t,c,d,\biggl(\frac{a}{p},\frac{b}{p}\biggl)\biggl)[t^0].
$$


In [17]:
## here we need b_1(0)=0
def log_beta_help(omega_p,p,betap):
    omega_p=omega_p+O(p)
    L=[[0,[1,0]]]
    for j in range (1,p^2-1):
        if j%(p+1)!=0:    ##checking if beta_p^j is in Q_p
            vec=pari.lindep([betap^j+O(p),1,omega_p+O(p)])
            L.append([j,[ZZ(-vec[1]/vec[0]+O(p)),ZZ(-vec[2]/vec[0]+O(p))]])
        else:
            vec=ZZ(betap^j+O(p))
            L.append([j,[vec,0]])
    return(L)


def log_beta(omega_p,c,d,p,l,betap):
    L=log_beta_help(omega_p,p,betap)
    total=0
    for j in range (0,len(L)):
        vecval=L[j]
        a1,b1=[vecval[1][0],vecval[1][1]]
        L_both=power_series_values_for_sum_smoothed_norm_0(c,d,2,l,a1/p,b1/p)  
        total=total+vecval[0]*QQ(sum_for_H_smoothed_value_at_k(1,c,d,2,l,L_both[0],L_both[1])) 
    if l==2:
        return((4*total)%(p^2-1))
    if l==3:
        return((3*total)%(p^2-1))
    return(total%(p^2-1))

####################### using the cocycle relation

def log_beta_low_level_cocycle(omega_p,c,d,p,l,betap,gamma):
    L=log_beta_help(omega_p,p,betap)
    total=0
    for j in range (0,len(L)):
        vecval=L[j]
        a1,b1=gamma^(-1)*vector([vecval[1][0],(vecval[1][1])])
        L_both=power_series_values_for_sum_smoothed_norm_0(c,d,2,l,a1/p,b1/p)  
        total=total+vecval[0]*QQ(sum_for_H_smoothed_value_at_k(1,c,d,2,l,L_both[0],L_both[1])) 
    if l==2:
        return((4*total)%(p^2-1))
    if l==3:
        return((3*total)%(p^2-1))
    return(total%(p^2-1))
    


def log_beta_cocycle(omega,Listmat,p,l,betap,T):
    c=l
    val=0
    for mat in Listmat:
        Mat=Matrix([[mat.a,mat.b],[mat.c,mat.d]])
        if Mat.det()!=0:
            Mat_minus=Mat^(-1)
            d=mat.x
            omega_p=T(omega)
            val+=log_beta_low_level_cocycle(omega_p,c,d,p,l,betap,Mat)*mat.coeff
    return(val%(p^2-1)) 



## Computing $\log_p$

Finally, we compute $\log_p(u_\ell^{\prime}(\omega))$.
$$\log_p(u_\ell^{\prime}(\omega))=12I_{c,d,\omega}=\int_{\mathbb{Z}_p^2-p\mathbb{Z}_p^2}\log_p(x+\omega y)d\mu(c,d)(x,y).
$$
And for all $c>0,d$, we have
\begin{split}
I_{c,d,\omega}&=
\sum_{\substack{a,b\mod p\\a,b\neq 0,0}}\log_p(a+b\omega)h_\omega^{(\ell)}\biggl(t,c,d,\biggl(\frac{a}{p},\frac{b}{p}\biggl)\biggl)[t^0]\\
&+\sum_{\substack{a,b\mod p\\a,b\neq 0,0}}\sum_{n=1}^{M'}\frac{1}{n}\sum_{j=0}^n\binom{n}{j}\frac{(-1)^{j-1}}{(a+b\omega)^j}j!p^j h_\omega^{(\ell)}(t,c,d,(\frac{a}{p},\frac{b}{p}))[t^j] +O(p^{M+1})
\end{split}


Using the cocycle relation of $\mu$, we are left to compute integral of the form $I_{c,d,-\gamma^{-1}-(\omega)}$ for $c=\ell$ and $d\leq l-1$.

In [1]:

def sum_log_omega_low_level(omega_p,c,d,p,l):#compute the first sum
    val1=0
    for a in range(0,p):
        for b in range(0,p):
            if a+b!=0:
                L_values=[]
                L_both=power_series_values_for_sum_smoothed(c,d,2,l,a/p,b/p)
                val1+=log(a+omega_p*b)*QQ(sum_for_H_smoothed_value_at_k(c*omega_p+d,c,d,2,l,L_both[0],L_both[1]))
    return(val1)

def value_int_log_low_level(omega,c,d,p,l,M,T):## compute the second one, this takes much longer than the first one
    beta=c*omega+d
    omega_p=T(omega)
    val2=0
    for a in range(0,p):
        print('    ','computation for a=',a,'and b in' ,[0,p-1])
        for b in range(0,p):
            if a+b!=0:
                L_values=[]
                L_both=power_series_values_for_sum_smoothed(c,d,M+3,l,a/p,b/p)
                for k in range (2,M+3):##recall the shift of 2 for the coeff of t^k
                    normalization_factor=factorial(k-2)*p^(k-2)
                    L_values.append(normalization_factor*sum_for_H_smoothed_value_at_k(beta,c,d,k,l,L_both[0],L_both[1]))
                for n in range (1,M+1):
                    for j in range(0,n+1):
                        val2+=binomial(n,j)*(-1)^(j-1)*T(L_values[j])/(n*(a+b*omega_p)^j)
    return(val2)



def value_int_log_p_adic(omega,c,d,p,l,M,T):## call both functions 
    time1=cputime()
    omega_p=T(omega)
    val1=sum_log_omega_low_level(omega_p,c,d,p,l)
    val2=value_int_log_low_level(omega,c,d,p,l,M,T)
    valtot=val1+val2
    if l==2:
        valtot=4*valtot
    if l==3:
        valtot=3*valtot
    return(valtot)

###################################using the cocycle relation

def value_int_log_p_adic_cocycle(omega,Listmat,p,l,M,T): 
    c=l
    val1=0
    val2=0
    print('  ','We use the cocycle relation to compute the p-unit')
    for i in range (len(Listmat)):
        mat=Listmat[i]
        Mat=Matrix([[mat.a,mat.b],[mat.c,mat.d]])
        if Mat.det()!=0:
            print('  ','Computation for matrix',i+1,'/',len(Listmat))
            Mat_minus=Mat^(-1)
            d=mat.x ## mat.x gives i associated to a_{gamma,i}=mat.coeff in the decomposition of the modular symbol 
            omega_i=-(-omega*Mat_minus[0,0]+Mat_minus[0,1])/(-omega*Mat_minus[1,0]+Mat_minus[1,1])
            omega_p_i=T(omega_i)
            val1+=sum_log_omega_low_level(omega_p_i,c,d,p,l)*mat.coeff
            val2+=value_int_log_low_level(omega_i,c,d,p,l,M,T)*mat.coeff
    valtot=val1+val2
    if l==2:
        valtot=4*valtot
    if l==3:
        valtot=3*valtot
    return(valtot) 






## Putting everything together

In [21]:
## compute the unit
def get_u_omega(omega,c,d,p,l,M,betap,T):
    time1=cputime()
    val_log_p=value_int_log_p_adic(omega,c,d,p,l,M,T)
    val_log_beta=ZZ(log_beta(T(omega),c,d,p,l,betap))
    p_power=QQ(order_p(c,d,p,l))
    print(' value for',omega,' computed in ',cputime()-time1)
    return([(p^(p_power))*(val_log_p.exp())*T(betap)^val_log_beta,[val_log_p,val_log_beta,p_power]])





def get_u_omega_cocycle(omega,Listmat,p,l,M,betap,T,c_start,d_start):
    time1=cputime()
    val_log_p=value_int_log_p_adic_cocycle(omega,Listmat,p,l,M,T)
    val_log_beta=ZZ(log_beta_cocycle(omega,Listmat,p,l,betap,T))
    p_power=QQ(order_p_cocycle(omega,Listmat,p,l,c_start,d_start))
    print(' value for',omega,' computed in ',cputime()-time1)
    return([(p^(p_power))*(val_log_p.exp())*T(betap)^val_log_beta,[val_log_p,val_log_beta,p_power]])



In [20]:
def get_pol_u_omega(List_val,M1,p,T):## return the minimal polynomial using the values of all the conjugates
    listval_omega=[]
    for k in range(0,len(List_val)):
        listval_omega.append(List_val[k][0])
    pol=1
    for u_omega in listval_omega:
        pol=pol*(x-u_omega)*(x-u_omega^(-1))
    listcoeff=pol.coefficients()
    listpower=pol.exponents()
    polK=0
    polK_v2=0
    for k in range(0,len(listcoeff)):## we try to recognize the coefficients using 2 methods, one using LLL and the other using Sage  
        valcoeff_p=listcoeff[k]
        coefficients=(valcoeff_p.polynomial()).coefficients()
        if len(coefficients) == 1:
            coefficients.append(T(0))
        x0,xd=coefficients
        x0val=x0.valuation()
        xdval=xd.valuation()
        p_deno_x0=p^(max(0,-x0val))
        p_deno_xd=p^(max(0,-xdval))
        find_x1=[]
        find_x2=[]
        find_x1=pari.lindep([p_deno_x0*x0,1])## we want to use lindep in Z_p
        find_x2=pari.lindep([p_deno_xd*xd,1])
        x1=-(find_x1[1]/find_x1[0])/p_deno_x0
        x2=-(find_x2[1]/find_x2[0])/p_deno_xd
        val_coeff=(QQ(x1)+QQ(x2)*d_K)
        polK+=x^(listpower[k])*val_coeff
        polK_v2+=x^(listpower[k])*K(valcoeff_p)
    if(polK==polK_v2):
        print('same pol with the 2 methods')
        print('for p=',p,'minimal polynomial is,',polK)
        return(polK)
    else:
        print('pol are not the same')
        print(polK,polK_v2)
        return([polK,polK_v2])
    



        

In [1]:
def computing_units(getgam,l,M1,p,T,betap): ## putting everything together
    M2=M1+floor(log(M1))## we need to compute extra terms in the series
    listval_omega=[]
    t1=cputime()
    for j in range (0,len(getgam)):
        omega=getgam[j][1]
        [c_start,d_start]=getgam[j][2][1]
        print('computation for omega =',omega,'and','(c,d)=', c_start,d_start,'(',j+1,'/',len(getgam),')')
        Listmat=getGammas(ZZ(d_start),ZZ(c_start))[:-1]
        if c_start<= len(Listmat)*l:## don't need to use the coycle relations when c is small enough
            List_u_info=get_u_omega(omega,c_start,d_start,p,l,M2,betap,T)
            listval_omega.append(List_u_info)
        else:
            List_u_info=get_u_omega_cocycle(omega,Listmat,p,l,M2,betap,T,c_start,d_start)
            listval_omega.append(List_u_info)
    print('final computation done in',cputime()-t1)
    pol=get_pol_u_omega(listval_omega,M1,p,T)
    return([pol,listval_omega])


